### Imports

In [ ]:
import torch
from tqdm import tqdm
import os
from torcheval.metrics import BinaryAUROC
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append("../utils")
from utils import load_model_and_tokenizer, load_prompts, apply_chat_template
from construct_predictors import collect_hidden_activations, Predictor

torch.set_grad_enabled(False)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

### Sparse LLaMA2

In [ ]:
model_path="../../models/SparseLLM/prosparse-llama-2-7b"
model, tokenizer = load_model_and_tokenizer(model_path, attn_implementation="eager", torch_dtype=torch.float16, device_map="cuda:0")
prompts = load_prompts("test_prompts.json")
prompts = apply_chat_template(prompts, tokenizer)
mlp_inputs_test, gate_outputs_test, down_inputs_test = collect_hidden_activations(model, tokenizer, prompts)


In [ ]:
mlp_inputs_test = torch.concat([x.unsqueeze(0) for x in mlp_inputs_test], dim=0).numpy()
gate_outputs_test = torch.concat([x.unsqueeze(0) for x in gate_outputs_test], dim=0).numpy()
down_inputs_test = torch.concat([x.unsqueeze(0) for x in down_inputs_test], dim=0).numpy()

In [ ]:
mlp_inputs_test_map = np.memmap("mlp_inputs_test.dat", dtype=np.float16, mode='w+', shape=mlp_inputs_test.shape)
gate_outputs_test_map = np.memmap("gate_outputs_test.dat", dtype=np.float16, mode='w+', shape=gate_outputs_test.shape)
down_inputs_test_map = np.memmap("down_inputs_test.dat", dtype=np.float16, mode='w+', shape=down_inputs_test.shape)
mlp_inputs_test_map[:] = mlp_inputs_test
gate_outputs_test_map[:] = gate_outputs_test
down_inputs_test_map[:] = down_inputs_test

del mlp_inputs_test, gate_outputs_test, down_inputs_test

In [ ]:
torch.cuda.empty_cache()
import gc
gc.collect()

In [ ]:
prompts = load_prompts("../predictors/calibration_prompts.json")
prompts = apply_chat_template(prompts, tokenizer)
mlp_inputs_train, gate_outputs_train, down_inputs_train = collect_hidden_activations(model, tokenizer, prompts)

In [ ]:
mlp_inputs_train = torch.concat([x.unsqueeze(0) for x in mlp_inputs_train], dim=0).numpy()
gate_outputs_train = torch.concat([x.unsqueeze(0) for x in gate_outputs_train], dim=0).numpy()
down_inputs_train = torch.concat([x.unsqueeze(0) for x in down_inputs_train], dim=0).numpy()
mlp_inputs_train_map = np.memmap("mlp_inputs_train.dat", dtype=np.float16, mode='w+', shape=mlp_inputs_train.shape)
gate_outputs_train_map = np.memmap("gate_outputs_train.dat", dtype=np.float16, mode='w+', shape=gate_outputs_train.shape)
down_inputs_train_map = np.memmap("down_inputs_train.dat", dtype=np.float16, mode='w+', shape=down_inputs_train.shape)
mlp_inputs_train_map[:] = mlp_inputs_train
gate_outputs_train_map[:] = gate_outputs_train
down_inputs_train_map[:] = down_inputs_train

del mlp_inputs_train, gate_outputs_train, down_inputs_train

In [ ]:
torch.cuda.empty_cache()
import gc
gc.collect()

In [ ]:
model=model.to("cpu")
intermediate_size, hidden_size=model.model.layers[0].mlp.gate_proj.weight.shape
n_layers = len(model.model.layers)
metric = BinaryAUROC(num_tasks=intermediate_size)

In [ ]:
rank = 256
layer_id = 0 

In [ ]:
import math
mu = 1

In [ ]:
n_selected_tokens_list = [0, 100, 1000,5000, 10000,15000, 20000,25000]
layer_id_list = [0,3,16,31]
aucs = [[] for _ in range(len(layer_id_list))]
device = "cuda"
for i, layer_id in enumerate(tqdm(layer_id_list)):
    weights = model.model.layers[layer_id].mlp.gate_proj.weight
    input_data = torch.tensor(mlp_inputs_test_map[layer_id]).to(device)
    true_values = model.model.layers[layer_id].mlp.act_fn(torch.tensor(gate_outputs_test_map[layer_id]).to(device))
    true_sparsity_pattern = (true_values > 0.0).to(torch.int16).T
    
    # baseline, naive svd, n tokens = 0
    u, s, v = torch.linalg.svd(weights.to(torch.float64))
    down_proj = (v * (s**0.0).unsqueeze(1))[:rank].to(torch.float16).to(device)
    up_proj = (u[:, :rank] * (s[:rank] ** 1.0).unsqueeze(0)).to(torch.float16).to(device)
    pred_values = (input_data @ down_proj.T @ up_proj.T).T
    metric.reset()
    metric.update(pred_values, true_sparsity_pattern)
    aucs[i].append(metric.compute().mean().item())
    
    
    # data-aware
    for n_selected_tokens in n_selected_tokens_list[1:]:
        X = torch.tensor(mlp_inputs_train_map[layer_id])[:n_selected_tokens].to(torch.float64)
        I = math.sqrt(mu) * torch.eye(X.shape[1], device=X.device, dtype=X.dtype)
        X= torch.cat([X, I], dim=0)
        S = X.T @ X
        S = torch.linalg.cholesky(S, upper=False)
        u, s, v = torch.linalg.svd(weights.to(torch.float64) @ S)
        v = torch.linalg.solve(S.T, v.T).T
        down_proj = (v * (s**0.0).unsqueeze(1))[:rank].to(torch.float16).to(device)
        up_proj = (u[:, :rank] * (s[:rank] ** 1.0).unsqueeze(0)).to(torch.float16).to(device)
        pred_values = (input_data @ down_proj.T @ up_proj.T).T
        metric.reset()
        metric.update(pred_values, true_sparsity_pattern)
        aucs[i].append(metric.compute().mean().item())


In [ ]:
ms = 7.0
plt.figure(figsize=(5,4))
colors = ["blue", "red", "green", "orange"]
markers = ["D","s","v", "^"]
names = ["First Layer", "Second Layer", "Middle Layer", "Last Layer"]
for i, layer_id in enumerate(layer_id_list):
    if i == 1:
        continue
    plt.plot(n_selected_tokens_list, aucs[i], f"{markers[i]}-", markersize=ms, label=f"{names[i]}", color=colors[i]) # 
plt.grid()

plt.xlabel("Number of Calibration Tokens")
plt.ylabel("ROC AUC Score")
# plt.ylim([None, 1.0])
plt.legend()
plt.tight_layout()
plt.savefig("analysis-calibration-data-for-svd.pdf", format="pdf")  

In [ ]:
def compute_predictor_bias(n_neurons: int, w_down: torch.Tensor, mlp_input: torch.Tensor, down_input: torch.Tensor, desired_sparsity: float):
    compute_dtype = torch.float64
    compute_device = "cpu"
    n_tokens = mlp_input.shape[0]# half of the dataset for S construction and other for bias calibration

    ### Bias calibration

    X = mlp_input[:].to(compute_device).to(compute_dtype) # (n_tokens, d)
    predicted_values = (X @ (down_proj.T @ up_proj.T)).T # (D, n_tokens)

    down_norms = torch.linalg.norm(w_down.to(compute_device).to(compute_dtype), dim=0, keepdim=True).T # (D, 1)
    neuron_importance = down_input[:].T.to(compute_device).to(compute_dtype) * down_norms # (D, n_tokens)

    sort_indices = torch.argsort(predicted_values, dim=-1) # (D, n_tokens)
    sorted_neuron_importance = torch.gather(neuron_importance, dim=-1, index=sort_indices).to(torch.float32) # (D, n_tokens)
    
    penalty = torch.cumsum(sorted_neuron_importance**2, dim=-1) # (D, n_tokens)

    target_size = 32
    indices = torch.linspace(0, penalty.shape[-1] - 1, target_size).to(torch.int64)
    multiplier = penalty.shape[-1] // target_size
    penalty = penalty[:, indices]  # (D, target_size)
    delta_penalty = penalty[:,1:]-penalty[:,:-1]
    delta_penalty = torch.concat([delta_penalty, torch.ones(n_neurons, 1)*float('inf')], dim=-1)

    # init thresholds
    thresholds = torch.clip(
        (penalty.cumsum(dim=-1) == 0).sum(dim=-1).to(torch.int64) - 1, 0, int(0.95*target_size) - 1,
    )

    sparsity = thresholds.to(torch.float16).mean().item() / target_size


    loss_increase = torch.gather(delta_penalty, dim=-1, index=thresholds.unsqueeze(1)).squeeze(1)
    print(f"Increasing sparsity from {100*sparsity:.1f} to {100*desired_sparsity:.1f}")
    n_steps = int((desired_sparsity - sparsity)/ (1/n_neurons/target_size))
    for _ in tqdm(range(n_steps)): # desired sparsity
        best_neuron = torch.argmin(loss_increase)
        thresholds[best_neuron] += 1
        loss_increase[best_neuron] = delta_penalty[best_neuron][thresholds[best_neuron]]

    sorted_predicted_values = torch.gather(predicted_values, dim=-1, index=sort_indices)
    bias = torch.gather(
        sorted_predicted_values, dim=-1, index=thresholds.unsqueeze(1) * multiplier
    ).squeeze(1)
    torch.cuda.empty_cache()
    return -bias

In [ ]:
import math
mu = 1

In [ ]:
n_selected_tokens_list = [0,1000,5000, 10000,15000, 20000,25000]
layer_id_list = [0,3,16,31]
sparsities = [[] for _ in range(len(layer_id_list))]
rel_errors = [[] for _ in range(len(layer_id_list))]
recalls = [[] for _ in range(len(layer_id_list))]
device = "cuda"
for i, layer_id in enumerate(tqdm(layer_id_list)):
    weights = model.model.layers[layer_id].mlp.gate_proj.weight
    input_data = torch.tensor(mlp_inputs_test_map[layer_id])
    true_values = model.model.layers[layer_id].mlp.act_fn(torch.tensor(gate_outputs_test_map[layer_id])).to(device)
    true_pattern = (true_values > 0.0)
    true_output = (true_values * (input_data.to(device) @ model.model.layers[layer_id].mlp.up_proj.weight.T.to(device))) @ model.model.layers[layer_id].mlp.down_proj.weight.T.to(device)
    
    # data-aware
    X = torch.tensor(mlp_inputs_train_map[layer_id])[:].to(torch.float64)
    I = math.sqrt(mu) * torch.eye(X.shape[1], device=X.device, dtype=X.dtype)
    X= torch.cat([X, I], dim=0)
    S = X.T @ X
    S = torch.linalg.cholesky(S, upper=False)
    u, s, v = torch.linalg.svd(weights.to(torch.float64) @ S)
    v = torch.linalg.solve(S.T, v.T).T
    down_proj = (v * (s**0.0).unsqueeze(1))[:rank]
    up_proj = (u[:, :rank] * (s[:rank] ** 1.0).unsqueeze(0))
    
    pred_unbiased = input_data.to(torch.float16).to(device) @ down_proj.T.to(torch.float16).to(device) @ up_proj.T.to(torch.float16).to(device)
    
    # baseline
    bias = torch.zeros(weights.shape[0])
    pred_pattern = ( pred_unbiased + bias.to(torch.float16).to(device)).T > 0.0
    pred_output = (pred_pattern.T.to(device) * true_values * (input_data.to(device) @ model.model.layers[layer_id].mlp.up_proj.weight.T.to(device))) @ model.model.layers[layer_id].mlp.down_proj.weight.T.to(device)
    sparsities[i].append((1-pred_pattern.to(torch.float16).mean().item())*100)
    rel_errors[i].append(torch.mean(torch.linalg.norm(true_output-pred_output, dim=-1)/(torch.linalg.norm(true_output, dim=-1)+1e-6)).item()*100)
    print("sparsity=", sparsities[i])
    print("error=", rel_errors[i])
    for n_selected_tokens in n_selected_tokens_list[1:]: 
        bias = compute_predictor_bias(n_neurons=weights.shape[0],
                                      w_down=model.model.layers[layer_id].mlp.down_proj.weight,
                                      mlp_input=torch.tensor(mlp_inputs_train_map[layer_id][:n_selected_tokens]).to("cpu"), 
                                      down_input=torch.tensor(down_inputs_train_map[layer_id][:n_selected_tokens]),
                                      desired_sparsity=0.5
                                      )    
        pred_pattern = ( pred_unbiased + bias.to(torch.float16).to(device)).T > 0.0
        pred_output = (pred_pattern.T.to(device) * true_values * (input_data.to(device) @ model.model.layers[layer_id].mlp.up_proj.weight.T.to(device))) @ model.model.layers[layer_id].mlp.down_proj.weight.T.to(device)
  
        sparsities[i].append((1-pred_pattern.to(torch.float16).mean().item())*100)
        rel_errors[i].append(torch.mean(torch.linalg.norm(true_output-pred_output, dim=-1)/(torch.linalg.norm(true_output, dim=-1)+1e-6)).item()*100)
        print("sparsity=", sparsities[i])
        print("error=", rel_errors[i])


In [ ]:
ms = 7.0
plt.figure(figsize=(5,4))
n_selected_tokens_list = [0,1000,5000, 10000,15000, 20000,25000]
colors = ["blue", "red", "green", "orange"]
markers = ["D","s","v", "^"]
names = ["First Layer", "Second Layer", "Middle Layer", "Last Layer"]
for i, layer_id in enumerate(layer_id_list):
    if i == 1:
        continue
    plt.plot(n_selected_tokens_list[1:], rel_errors[i][1:], f"{markers[i]}-", markersize=ms, label=f"{names[i]}", color=colors[i]) # 
plt.grid()

plt.xlabel("Number of Calibration Tokens")
plt.ylabel("FFN Relative Error, %")
# plt.ylim([None, 1.0])
plt.legend()
plt.tight_layout()
plt.savefig("analysis-calibration-data-for-bias.pdf", format="pdf")  